In [ ]:
using Kinbiont
using Plots
using CSV, DataFrames
using Statistics
using DelimitedFiles
using Random
using DecisionTree
using AbstractTrees
using MLJDecisionTreeInterface
using TreeRecipe

In this example we consider the result of the fit with Richards model of N.soli. The data are from  ["High-throughput characterization of bacterial responses to complex mixtures of chemical pollutants"](https://www.nature.com/articles/s41564-024-01626-9).


We start importing the fit results and the feature matrix:

In [ ]:
Kinbiont_res_test = readdlm("/df_for_ML/res_clean_ML_richards.csv", ',');
annotation_test = readdlm("df_for_ML/annotation_clean_richards.csv", ',');

We read in the annotation the order of the strains and we set the feature names

In [ ]:
ordered_strain = annotation_test[:, end];
feature_names = unique(annotation_test[1, 2:end])[2:(end-1)];

We select the parts of the annotation and results matrix that contains the N.soli

In [ ]:
index_strain = findall(s .== ordered_strain);
feature_matrix = annotation_test[index_strain, 2:(end-1)]

In [ ]:
Kinbiont_results = Kinbiont_res_test[:, index_strain]

We set some parameter of the decion tree and we perdom the regression on growth rate for example (row 9 of the results). In this case we do not set the maxdepth to see the importance scores

In [ ]:
seed = Random.seed!(1234)
n_folds = 10
dt_gr = downstream_decision_tree_regression(Kinbiont_results,
            feature_matrix,
            9;
            do_pruning=false,
            pruning_accuracy=1.00,
            verbose=true,
            do_cross_validation=true,
            max_depth=-1,
            n_folds_cv=n_folds,
            seed=seed
)

The cross validation results, and importance score are stored in

In [ ]:
dt_gr[1]

We perform again the tree with depth 2 to visualize it better:

In [ ]:
dt_gr = downstream_decision_tree_regression(Kinbiont_results,
            feature_matrix,
            9;
            do_pruning=false,
            pruning_accuracy=1.00,
            verbose=true,
            do_cross_validation=true,
            max_depth=2,
            n_folds_cv=n_folds,
            seed=seed
)


wt = DecisionTree.wrap(dt_nmax[1], (featurenames = feature_names,));
 
 
p2 = Plots.plot(wt, 0.9, 0.2; size = (900,400), connect_labels = ["yes", "no"]);
display(p2)